# Data Decomposition
created by Adam S. Charles and Esther M. Whang for CSHL2025

This exercise will walk you through some basics of data decompositions. We will be thinking of our data from the perspective of frequency transforms and matrix decompositions.

This notebook will provide hints and guidance but will leave the code writing up to you. We highly recommend attempting the code on your own before looking at the solutions or referring to aides such as ChatGPT. Struggling through the code will help in learning the concepts.

We would also like to recommend reading the documentation provided by the respective packages when using new functions -- numpy and scikit-learn have excellent documentation with examples.



In [ ]:
# Let's first load in all the packages you will need

## I like to keep all the imports at the top of the file
from skimage import io  # skimage is a package for dealing with images. io = in/out
import numpy as np      # numpy is the core package for numerical processing
import scipy.io
import h5py             # This package allows you to access newer MATLAB saves that are in HDF5 formats
from matplotlib import pyplot   as plt
import plotly.express as px


In [ ]:
## Let's first load the data

matFileData = h5py.File('/home/aaa_shared/cshl2025/Data/Somatic/SEUDO_J122_2015-11-16_L01_c01_b27.mat', mode='r')

matFileData['dFF'].shape
dataMovie = np.array(matFileData['dFF'])


#We are not going to be working with all frames of this dataset for the class, so let's just look at the first 1000 frames
dataMovie_small = dataMovie[0:1000,:,:]
dMshape   = dataMovie_small.shape # Keep the orignial size!!
dMshape

In [ ]:
frame_number = 0 
plt.imshow(dataMovie_small[frame_number,:,:])        
color_bar = plt.colorbar()
color_bar.set_label('Intensity')
plt.title("Example Frame from dataMovie_small")

In [ ]:
# Let's look at the histogram

number_of_bins = 2000 # determines the number of bins in the histogram
cnts = plt.hist(dataMovie_small.ravel(),bins=number_of_bins)

# labels
plt.title("Histogram of Intensity Values for Video")
plt.ylabel("Count")
plt.xlabel("Intensity Value")

### Transforms

One great way to understand a signal is to consider its frequency. 

During our course lectures, we discussed Fourier Transforms -- now you'll have a chance to use them on your own! We also touch upon its close cousin, the Wavelet transform, which you can explore for extra credit.

In this section, we will be looking at the frequency transform of a single pixel.

In [ ]:
# Let's start by looking at a summary image of some dataset. 

# TODO: Look at dataMovie -- what are some quick ways we can identify iteresting parts of our data?
# Hint: the CSHL2025_basics notebook is a great reference for this section
# CODE: selectSummary = "fano"   # Try switching between "mean", "median", "max" and "fano"
selectSummary = "fano" 
match selectSummary:
    case "mean":
        summaryImage = dataMovie_small.mean(axis=0) 
    case "max":
        summaryImage = dataMovie_small.max(axis=0)
    case "median":
        summaryImage = np.median(dataMovie_small,axis=0)
    case "fano":
        summaryImage = 20*(dataMovie_small.mean(axis=0)*2)/np.var(dataMovie_small,axis=0) #the 20 is to normalize


plt.imshow(summaryImage)
color_bar = plt.colorbar()
color_bar.set_label('Intensity')



In [ ]:
# Now, let's identify which pixels we found the most interesting.

# First, let's just look at a single pixel. We will look at a cropped image further along the notebook.


# TODO: a) Pick a single pixel that you feel has a lot of signal (look at the movie!). 
# Save the value of this pixel over all frames
lots_of_signal_pixel_trace = None # (replace None with the proper code)

# TODO: b) Pick a pixel that you feel has minimal signal
# Save the value of this pixel over all frames
minimal_signal_pixel_trace = None # (replace None with the proper code)

In [ ]:
#### Let's plot the two signals
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(range(dataMovie_small.shape[0])), y=lots_of_signal_pixel_trace,
                    mode='lines',name='signal'))
fig.add_trace(go.Scatter(x=np.array(range(dataMovie_small.shape[0])), y=minimal_signal_pixel_trace ,
                    mode='lines',name='noise'))
fig.show()

In [ ]:
#### Fourier Transform!!!

# Before the next section, we will be providing a quick demo of how to perform a Fourier Transform with Python

import matplotlib.pyplot as plt
import numpy as np

Fs = 100 # sampling rate
f1 = 3 # frequency of the signal
f2 = 7
x =  np.arange(0, 1, 1/Fs)
y = np.sin(2 * np.pi * f1 * x ) + np.sin(2 * np.pi * f2 * x )

# We have created a sine wave with a frequency of 5
# Now let's take the Fourier Transform using numpy.fft.fft!
y_fourier = np.fft.fftshift(np.fft.fft(y)/len(y)) 
# note: using fftshift centers the zeroth frequency component, which is useful when visualizing
x_freq = np.fft.fftshift(np.fft.fftfreq( len(y), d=1/Fs))

# showing the exact location of the samples
plt.figure()
plt.subplot(1, 2, 1)
plt.stem(x,y, 'r', )
plt.plot(x,y)
plt.title("Sine Wave")

plt.subplot(1, 2, 2)
plt.plot(x_freq, abs(y_fourier))
plt.xlim([-(max(f1,f2)+3), (max(f1,f2)+3)])

plt.title("Fourier Transform of Sine Wave")

In [ ]:
# Now that we've seen a demo of Fourier Transform, 
# let's look at the frequency transform of our selected pixels. 
# TODO: c) Compute the Fourier transforms of both single-pixel time traces. Reference the 

# We recommend using numpy.fft, which has documentation 
# https://numpy.org/doc/stable/reference/generated/numpy.fft.fft.html
# The example above will also be helpful

signalFFT = None # Take the Fourier transform of the signal trace
noiseFFT  = None # Take the Fourier transform of the noise trace

In [ ]:
# Often we plot the "power" of the data which corresponds to 
# the magnitude of the different frequency components. This is 
# done with the np.abs() function which takes the absolute value
# of an imaginary number (sqrt(x^2 + y^2) for x + j*y)

### let's plot the results of the Fourier Transformed signal
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(range(len(signalFFT))), y=np.log(np.abs(signalFFT)),
                    mode='lines',name='signal'))
fig.add_trace(go.Scatter(x=np.array(range(len(noiseFFT))), y=np.log(np.abs(noiseFFT)),
                    mode='lines',name='noise'))
fig.show()

# For extra credit, look up PyWavelets and compute a wavelet transform as well

In [ ]:
# TODO: d) Repeat the above for time traces, but this time, 
# instead of a single pixel, take the average over a image crop of a cell that does have a signal
# (Hint: remember the CSHL2025_basics notebook!)
lots_of_signal_avg_trace = None # replace None with code
# Also take the average over an area that doesn't have signal (a "blank" area)
blank_area_avg_trace = None # replace None with code

# TODO: Plot these results as well! Refer to the previous blocks of code
# CODE: 

In [ ]:
#########################################################################

# Questions:
# 1) What do you notice about the Fourier transform values?
# 2) How do the Fourier representations change between both time-traces? 


### Data-driven Decompositions
We will be exploring three different data decomposition methods: PCA, NMF, and ICA. Each have their own strength and weaknesses, which will be exploring with the scikit-learn package.

#### PCA

In [ ]:
import sklearn as skl
import sklearn.decomposition as dcmp

## Let's start with PCA

# Hint: This data is 3D!? Try running #    PCA using different numbers of principal components.  
#      - Hint: look up sklearn.decomposition.PCA (which you call as dcmp.PCA())

# a) Start with PCA: Take the first 200 frames of the SEUDO dataset from
#    yesterday and perform PCA on it.

# first, grab the first 200 frames
dataMovie_first200frames = dataMovie_small[0:200,:,:]

In [ ]:
# Now, let's consider how to perform PCA. We will be utilizing the 
# sklearn.decomposition.PCA function. Take a moment to look through the documentation on this function
# https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
# The bottom of the documentation page has some example code - spend a couple of minutes running them with our video data!

# If you run into problems when you replace X with our dataMovie_first200frames, you're on the right track

In [ ]:
##### Continue after testing with the code on the documentation!!



# Spoilers after this!




# You have been warned!




#########


In [ ]:
##### Now that you've looked at the documentation

# From testing sklearn.decomposition.PCA, you can see that it only takes in 2D data.
# However, our video is 3D: [Frames] x [Image Height] x [Image Width]
print(dataMovie_first200frames.shape)

# We need to figure out how to reshape our data into a 2D structure

# TODO: Reshape using np.reshape our dataMovie so that our video data is now a 2D matrix
# Hint: Think about our lecture - what are the two features of a video that we care about?
new_shape_dimension = (None,None*None) # (replace None with the the dimensions you want for your 2D matrix)
dataMovie_2D = np.reshape(dataMovie_first200frames, new_shape_dimension)


In [ ]:
# Now create a PCA object that takes the first 50 components of the data
# Now that we have the data reshaped, you can go back to the example code from the documentation!

n_components = 50

# TODO: use dcmp.PCA on the reshaped data!
dataPCA = None # call dcmp.PCA
# and any additional code you need to get your components

In [ ]:
# Now let's look at the PCs.
# First, note that the PCs are 2D, with the first dimension being n_components
print(np.shape(dataPCA.components_))
# To look at the PCAs as an image, we need to reshape this into a 3D shape!

# TODO: Reshape the PCA components
PCA_compImgs = np.reshape(None,(n_components,None, None))

# Now let's take a look at the spatial PC components:
fig = px.imshow(PCA_compImgs, animation_frame=0, binary_string=True, labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
#########################################################################

# Questions for PCA:
# 1) What do the spatial components look like?
# 2) What does the first element of the PCA output look like? The second? The last?
# 3) How can we look at the temporal components? Hint: Look at pca.transform() from scikit-learn

#### NMF

In [ ]:
# b) Now use a version of matrix decompositions we didn't cover: NMF. 
#    NMF stands for non-negative matrix decompositions. It's the basis for
#    some of the more well known decomposition algorithms. Use this on the
#    same data as above. 

# The way you call the NMF function is similar to how you use PCA. 
# https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html

# However, we do have one difference - we don't want negative values in our data matrix
dataMovie_2D_nmf = dataMovie_2D 
dataMovie_2D_nmf[dataMovie_2D<0] = 0 # This causes all negative values to become 0

In [ ]:
# TODO: Try the sklearn.decomposition.NMF! Create a NMF object that takes the first 50 components of the data
# Follow similar steps as PCA: make the object, fit, then reshape to look at the individual components

dataNMF   = None

#del dataMovie_2D_nmf # if you're running out of space, we suggest cleaning up as you go

NMF_compImgs = None # Reshape the components as a 3D matrix

# Now let's take a look at the spatial NMF components:
fig = px.imshow(NMF_compImgs, animation_frame=0, binary_string=True, labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
#########################################################################

# Questions for NMF:
# 1) What do the spatial components look like?
# 2) What does the first element of the NMF output look like? The second? The last?
# 3) How does it differ from the PCA components?

#### ICA

In [ ]:
# c) Finally, try out ICA! Look through the sklearn.decomposition.FastICA
# https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.FastICA.html

# Because the structure of the code is very similar to using PCA and NMF, 
# We will leave it up to use the documentation to complete this section

# TODO: Calculate ICA for your video data!

In [ ]:

# Now let's take a look at the spatial ICA components:
ICA_compImgs = None
fig = px.imshow(ICA_compImgs, animation_frame=0, binary_string=True, labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
# Questions for ICA:
# 1) What do the spatial components look like?
# 2) What does the first element of the ICA output look like? The second? The last?
# 3) How does it differ from the PCA and NMF components?

In [ ]:
# Questions for Data-Driven Decompositions
# 1) What does the first element of the PCA output look like? The second? The last?
# 2) How do the outputs of the three algorithms compare? 
# 3) Try to "align" the outputs by matching up ouputs (decomposition vectors)
#    between the threen algorithms.

In [ ]:
# EXTRA CREDIT:
# How might these decompositions be useful in denoising? 
# If you implement PCA with np.linalg.svd function (singular value decomposition), it's not too difficult to denoise the video
# This would be essentially manually performing PCA

# Try to see if you can use np.linalg.svd to reconstruct a denoised video!
